In [2]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import glob

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 35
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## Climate Laws Pipeline

**Source:** Climate Change Laws of the World (LSE Grantham / Climate Policy Radar)
**Access:** Manual CSV download (free registration form); auto-detects file in Downloads
**Download instructions:** See `docs/instructions_data_maintenance.md` — CLIMATE_LAWS section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Count / presence of climate laws and policies | Environmental/climate governance | Primary tier 2 |

### Note
Document-level dataset (one row per law/policy/litigation document). Aggregated to
country-year counts. Specific framework metric (stock of laws, recent flow, etc.) to be
decided at metric pass.

In [4]:
import pandas as pd
import os
import glob
from datetime import datetime

# Auto-detect the Climate Laws CSV in Downloads — no hardcoded filename/date
cl_pattern = os.path.join(DOWNLOADS_DIR, "Document_Data_Download*.csv")
cl_files = glob.glob(cl_pattern)

if not cl_files:
    print(f"No Climate Laws CSV found in {DOWNLOADS_DIR}")
    print("Download from climate-laws.org (data download request form)")
else:
    cl_file = max(cl_files, key=os.path.getmtime)
    print(f"Found: {os.path.basename(cl_file)}")

    cl_raw = pd.read_csv(cl_file, low_memory=False)
    print(f"\nShape: {cl_raw.shape}")
    print(f"Columns: {list(cl_raw.columns)}")

Found: Document_Data_Download-2026-06-16.csv

Shape: (12907, 36)
Columns: ['Document ID', 'Document Title', 'Family ID', 'Family Title', 'Family Summary', 'Collection Title(s)', 'Collection Description(s)', 'Document Variant', 'Document Content URL', 'Language', 'Source', 'Geography ISOs', 'Geographies', 'First event in timeline', 'Last event in timeline', 'Full timeline of events (types)', 'Full timeline of events (dates)', 'Date Added to System', 'Last Modified on System', 'Internal Document ID', 'Internal Family ID', 'Internal Corpus ID', 'Internal Collection ID(s)', 'Framework', 'Topic/Response', 'Hazard', 'Sector', 'Keyword', 'Instrument', 'Author', 'Author Type', 'Document URL', 'Family URL', 'Document Role', 'Document Type', 'Category']


In [12]:
import re

# National-only: count each country's own nationally-recorded climate laws.
# EU-level (EUR) documents are DROPPED — attributing them to members AND counting members'
# national transpositions would double-count the same policy, and the data has no reliable
# flag to identify transpositions. National records capture most transposed EU law anyway.

# Exclude UNFCCC (international reporting, not domestic governance); keep Legislative + Executive
cl = cl_raw[cl_raw['Category'].isin(['Legislative', 'Executive'])].copy()

# Deduplicate to family level (variant documents share a Family ID) so one law counts once
cl = cl.sort_values('First event in timeline').drop_duplicates(subset='Family ID', keep='first')

# Extract enactment year; coerce bad dates to NaT
cl['year'] = pd.to_datetime(cl['First event in timeline'], errors='coerce', utc=True).dt.year

# Drop malformed/missing years. 1900 is a fixed plausibility floor (no climate law predates it),
# not a data vintage — it never needs updating.
n_before = len(cl)
cl = cl[cl['year'].notna() & (cl['year'] >= 1900)].copy()
cl['year'] = cl['year'].astype(int)
print(f"Dropped {n_before - len(cl)} laws with missing/implausible dates")

def geo_to_iso3_list(geo_cell):
    """Resolve a Geography ISOs cell to sovereign ISO3 codes (national-only).
    Split multi-geography; keep 3-letter sovereign tokens; drop EU-level (EUR) and
    subnational tokens (e.g. BR-XX)."""
    if not isinstance(geo_cell, str):
        return []
    out = []
    for tok in re.split(r'[;,]', geo_cell):
        tok = tok.strip()
        if len(tok) == 3 and tok.isalpha() and tok != 'EUR':   # ISO3 sovereign only; EUR dropped
            out.append(tok)
    return out

# Expand each family-level law to its sovereign country list. Keep Family ID so distinct laws
# in the same country-year are NOT collapsed.
law_rows = []
for _, r in cl.iterrows():
    for iso3 in geo_to_iso3_list(r['Geography ISOs']):
        law_rows.append({'country_code': iso3, 'year': r['year'], 'family_id': r['Family ID']})

# Dedup only on (country, family) — never on (country, year)
laws = pd.DataFrame(law_rows).drop_duplicates(subset=['country_code', 'family_id'])

print(f"Family-level laws after filtering: {len(cl)}")
print(f"Country-law rows after expansion (national-only): {len(laws)}")
print(f"Countries: {laws['country_code'].nunique()}")
print(f"Year range: {laws['year'].min()} — {laws['year'].max()}")
print(f"\nHighest law-count countries (all years, distinct laws):")
print(laws.groupby('country_code').size().sort_values(ascending=False).head(10))

Dropped 5 laws with missing/implausible dates
Family-level laws after filtering: 6293
Country-law rows after expansion (national-only): 6162
Countries: 199
Year range: 1957 — 2026

Highest law-count countries (all years, distinct laws):
country_code
BRA    217
FRA    156
CHN    136
AUS    120
DNK    113
GBR    113
CAN    104
MEX    103
KOR    101
ESP    100
dtype: int64


In [13]:
# Build cumulative-stock panel: for each country-year, count laws enacted that year or earlier.
# This is the stock of climate laws/policies in force, the framework's climate-governance measure.

# Count NEW laws per country-year (distinct families enacted that year)
new_per_year = laws.groupby(['country_code', 'year']).size().reset_index(name='new_laws')

# Build a complete country-year grid from each country's first law year to the latest data year,
# so cumulative stock carries forward in years with no new laws. Latest year derived from data.
latest_year = int(laws['year'].max())
grid_rows = []
for cc, g in new_per_year.groupby('country_code'):
    start = int(g['year'].min())
    for y in range(start, latest_year + 1):
        grid_rows.append({'country_code': cc, 'year': y})
grid = pd.DataFrame(grid_rows)

# Merge new-law counts onto the grid, fill gaps with zero, cumulate within country
panel = grid.merge(new_per_year, on=['country_code', 'year'], how='left')
panel['new_laws'] = panel['new_laws'].fillna(0).astype(int)
panel = panel.sort_values(['country_code', 'year'])
panel['climate_laws_cumulative'] = panel.groupby('country_code')['new_laws'].cumsum()

# Filter to framework start year (keep cumulative stock, which already reflects pre-start laws)
panel = panel[panel['year'] >= FRAMEWORK_START_YEAR].copy()
panel = panel[['country_code', 'year', 'new_laws', 'climate_laws_cumulative']].reset_index(drop=True)

print(f"Panel shape: {panel.shape}")
print(f"Countries: {panel['country_code'].nunique()}")
print(f"Years: {panel['year'].min()} — {panel['year'].max()}")
print(f"\nHighest cumulative stock (latest year):")
latest = panel[panel['year'] == panel['year'].max()]
print(latest.sort_values('climate_laws_cumulative', ascending=False).head(10).to_string(index=False))

Panel shape: (5268, 4)
Countries: 199
Years: 1990 — 2026

Highest cumulative stock (latest year):
country_code  year  new_laws  climate_laws_cumulative
         BRA  2026        17                      217
         FRA  2026         2                      156
         CHN  2026         2                      136
         AUS  2026         0                      120
         DNK  2026         1                      113
         GBR  2026         1                      113
         CAN  2026         1                      104
         MEX  2026         0                      103
         KOR  2026         1                      101
         ESP  2026         0                      100


In [14]:
# Data currency: latest year present in the data — no hardcoding
data_as_of = str(int(laws['year'].max()))

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "climate_laws_clean.csv")
panel.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {panel.shape}")

n_countries = panel['country_code'].nunique()

# Update download log
update_entry(
    "CLIMATE_LAWS",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of,
    local_filename="climate_laws_clean.csv",
    latest_available_version=data_as_of,
    notes=("Cumulative stock of domestic climate laws/policies per country-year (and new_laws flow). "
           "Source: Climate Change Laws of the World (LSE Grantham / Climate Policy Radar), manual CSV, "
           "auto-detected in Downloads. UNFCCC category excluded (international reporting, not domestic "
           "governance); Legislative + Executive kept. Deduplicated to Family ID (variant docs collapsed). "
           "NATIONAL-ONLY: EU-level (EUR) documents dropped to avoid double-counting EU law with member "
           "national transpositions; subnational tokens (e.g. BR-XX) dropped, national code retained. "
           f"Distinct laws counted per (country, family). Coverage: {n_countries} countries — good.")
)
print_entry("CLIMATE_LAWS")

Written: C:\Users\mjbou\governance-framework\data\processed\climate_laws_clean.csv
Shape: (5268, 4)
[download_log] Updated entry for CLIMATE_LAWS
  source_id: CLIMATE_LAWS
  last_attempted_date: 2026-06-17
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2026
  local_filename: climate_laws_clean.csv
  latest_available_version: 2026
  no_update_reason: nan
  notes: Cumulative stock of domestic climate laws/policies per country-year (and new_laws flow). Source: Climate Change Laws of the World (LSE Grantham / Climate Policy Radar), manual CSV, auto-detected in Downloads. UNFCCC category excluded (international reporting, not domestic governance); Legislative + Executive kept. Deduplicated to Family ID (variant docs collapsed). NATIONAL-ONLY: EU-level (EUR) documents dropped to avoid double-counting EU law with member national transpositions; subnational tokens (e.g. BR-XX) dropped, national code retained. Distinct laws counted per (country, family). Coverage: 199 countries 